# Aula 05 — Distribuição Normal e Teorema Central do Limite

## Objetivos
1. Dominar a **Distribuição Normal** e seus parâmetros.
2. Padronização (z-score).
3. Entender e demonstrar o **Teorema Central do Limite (TCL)**.
4. Verificar normalidade com Q-Q plot e teste de Shapiro-Wilk.

---

## 1. Distribuição Normal $N(\mu, \sigma^2)$

A distribuição mais importante da estatística. Densidade:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

Propriedades:
- Simétrica em torno de $\mu$.
- Forma de sino.
- Definida por **dois parâmetros**: média $\mu$ e desvio-padrão $\sigma$.

### Regra empírica (68-95-99,7)

Em uma Normal:
- $\approx 68\%$ dos dados estão em $[\mu - \sigma,\ \mu + \sigma]$
- $\approx 95\%$ em $[\mu - 2\sigma,\ \mu + 2\sigma]$
- $\approx 99{,}7\%$ em $[\mu - 3\sigma,\ \mu + 3\sigma]$

### Z-score (padronização)
$$Z = \frac{X - \mu}{\sigma} \sim N(0, 1)$$

Útil para comparar valores de distribuições diferentes ou aplicar tabelas.

---

## 2. Teorema Central do Limite

> Seja $X_1, X_2, \ldots, X_n$ uma amostra aleatória de qualquer distribuição com média
> $\mu$ e variância $\sigma^2 < \infty$. Então, quando $n \to \infty$:
> $$\bar{X}_n \approx N\!\left(\mu,\ \frac{\sigma^2}{n}\right)$$

**Por que isso é mágico?**
Não importa a forma da distribuição original — a **média amostral** tende à Normal.
Isso permite construir intervalos de confiança e testes de hipótese mesmo sem conhecer
a distribuição dos dados.

Na prática: $n \ge 30$ costuma ser suficiente.

In [ ]:
# === SETUP PARA GOOGLE COLAB ===
# Esta célula garante que o notebook funcione tanto em ambiente local
# quanto em quem clonar o repositório direto no Colab.

import sys, os, subprocess

# 1) Instala dependências (silencioso se já existirem)
pkgs = ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "requests", "plotly", "statsmodels"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# 2) Se estivermos no Colab e o repo ainda não foi clonado, clona.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/220719/curso-estatistica-python.git"  # ajuste após criar o repo

if IN_COLAB and not os.path.exists("/content/curso-estatistica-python"):
    subprocess.run(["git", "clone", REPO_URL, "/content/curso-estatistica-python"], check=False)

# 3) Adiciona o diretório utils ao PYTHONPATH para importar o módulo SIDRA
BASE = "/content/curso-estatistica-python" if IN_COLAB else ".."
if BASE not in sys.path:
    sys.path.append(BASE)

# 4) Pasta de gráficos (cria se não existir)
GRAFICOS_DIR = os.path.join(BASE, "graficos")
os.makedirs(GRAFICOS_DIR, exist_ok=True)

print("✓ Ambiente pronto. Pasta de gráficos:", GRAFICOS_DIR)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(2026)

## 3. Visualizando a Normal e a regra 68-95-99,7

In [ ]:
mu, sigma = 0, 1
N = stats.norm(mu, sigma)
x = np.linspace(-4, 4, 400)

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(x, N.pdf(x), color="navy", linewidth=2)

# Regiões de 1, 2, 3 sigma
for k, cor, alpha in [(1, "#74c476", 0.5), (2, "#fd8d3c", 0.3), (3, "#e6550d", 0.2)]:
    mask = (x >= mu - k*sigma) & (x <= mu + k*sigma)
    ax.fill_between(x[mask], N.pdf(x[mask]), alpha=alpha, color=cor,
                    label=f"±{k}σ: {(N.cdf(mu+k*sigma) - N.cdf(mu-k*sigma))*100:.1f}%")
ax.set_title("Distribuição Normal padrão N(0,1) — Regra 68-95-99,7%", fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula05_normal_regra_empirica.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Padronização com dados reais

Vamos padronizar o IPCA mensal e ver onde caem os meses extremos.

In [ ]:
import pandas as pd
from utils.sidra_api import obter_ipca_mensal
df = obter_ipca_mensal(120)

mu_emp    = df["variacao_mensal"].mean()
sigma_emp = df["variacao_mensal"].std()

df["z"] = (df["variacao_mensal"] - mu_emp) / sigma_emp
df["categoria"] = pd.cut(df["z"].abs(),
                         bins=[0, 1, 2, 3, np.inf],
                         labels=["normal", "moderado", "extremo", "muito extremo"])
print(f"Média: {mu_emp:.3f}, σ: {sigma_emp:.3f}")
print(df["categoria"].value_counts())
df.tail()

## 5. Demonstrando o TCL na prática

Vamos partir de uma **distribuição exponencial** (muito assimétrica) e mostrar que
a distribuição das **médias amostrais** se aproxima da Normal conforme $n$ cresce.

In [ ]:
def simular_medias(distribuicao, tamanho_amostra, n_repeticoes=5000):
    """Sorteia n_repeticoes amostras de tamanho 'tamanho_amostra' e retorna as médias."""
    medias = np.empty(n_repeticoes)
    for i in range(n_repeticoes):
        amostra = distribuicao.rvs(size=tamanho_amostra, random_state=rng)
        medias[i] = amostra.mean()
    return medias

# População assimétrica: Exponencial(λ=1)
populacao = stats.expon(scale=1.0)

tamanhos = [1, 2, 5, 30, 100]
fig, axes = plt.subplots(1, len(tamanhos), figsize=(18, 4))

for ax, n in zip(axes, tamanhos):
    medias = simular_medias(populacao, n)
    sns.histplot(medias, bins=40, kde=True, ax=ax,
                 color="steelblue", edgecolor="white", stat="density")
    ax.set_title(f"n = {n}")
    ax.set_xlabel("Média amostral")
    ax.set_xlim(0, 4)

fig.suptitle("Teorema Central do Limite — médias de Exponencial(1)", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula05_tcl.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Observe: quando n cresce, a distribuição das médias fica simétrica e em forma de sino —")
print("mesmo partindo de uma população claramente assimétrica.")

## 6. Verificando normalidade

### 6.1 Q-Q Plot
Se os dados são Normais, os pontos seguem a diagonal.

### 6.2 Teste de Shapiro-Wilk
$H_0$: os dados vêm de uma Normal.
Rejeitamos $H_0$ se p-valor < 0,05.

In [ ]:
from scipy.stats import probplot, shapiro

ipca_vals = df["variacao_mensal"].values

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histograma com Normal teórica sobreposta
sns.histplot(ipca_vals, bins=20, stat="density", ax=axes[0],
             color="lightblue", edgecolor="white")
xx = np.linspace(ipca_vals.min(), ipca_vals.max(), 200)
axes[0].plot(xx, stats.norm.pdf(xx, mu_emp, sigma_emp),
             color="darkred", linewidth=2, label="Normal ajustada")
axes[0].set_title("IPCA mensal vs Normal teórica")
axes[0].legend()

# Q-Q plot
probplot(ipca_vals, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot vs Normal")
axes[1].get_lines()[0].set_color("steelblue")
axes[1].get_lines()[1].set_color("red")

plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula05_qqplot_ipca.png"), dpi=150, bbox_inches="tight")
plt.show()

stat, p = shapiro(ipca_vals)
print(f"Shapiro-Wilk: W = {stat:.4f}, p-valor = {p:.4f}")
print("Conclusão:", "Rejeita-se H0 → NÃO é Normal" if p < 0.05 else "Não se rejeita H0 → compatível com Normal")

## Exercícios

1. Sorteie 1000 amostras de tamanho $n=50$ de uma distribuição Uniforme(0, 1) e mostre
   que as médias seguem aproximadamente $N(0{,}5,\ 1/(12 \cdot 50))$.
2. Calcule o z-score do mês com maior IPCA dos últimos 10 anos. Em quantos desvios-padrão
   ele está da média?
3. Faça o teste de Shapiro-Wilk para o IPCA **apenas dos últimos 24 meses**. O resultado
   muda em relação aos 120 meses?

**Próxima aula:** Intervalos de Confiança.